In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda"
os.makedirs(f"{PROJECT_ROOT}/data/processed", exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda


In [6]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

df = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/trip_feature_store.parquet")
df["departure_datetime_local"] = pd.to_datetime(df["departure_datetime_local"])
df = df.sort_values(["route_id","departure_datetime_local"]).reset_index(drop=True)

df.head()

,company,trip_id,route_id,origin_port,dest_port,departure_datetime_local,date,dep_time,weekday,month,...,pax_lag_1w,pax_lag_2w,veh_lag_1w,veh_lag_2w,pax_roll_mean_20,pax_roll_std_20,veh_roll_mean_20,veh_roll_std_20,delay_roll_mean_20,occ_pax_roll_mean_20
0,LevanteFerries,T0000075,DEN-FOR,Denia,Formentera,2024-01-07 08:00:00,2024-01-07,08:00,6,1,...,542.0,533.0,194.0,160.0,581.25,157.994295,173.80,44.684390,5.60,0.645833
1,LevanteFerries,T0000076,DEN-FOR,Denia,Formentera,2024-01-07 12:00:00,2024-01-07,12:00,6,1,...,631.0,528.0,214.0,162.0,587.10,156.085638,175.95,43.950301,5.60,0.652333
2,LevanteFerries,T0000077,DEN-FOR,Denia,Formentera,2024-01-07 17:00:00,2024-01-07,17:00,6,1,...,591.0,399.0,168.0,114.0,585.10,155.248494,174.35,43.221918,6.40,0.650111
3,LevanteFerries,T0000078,DEN-FOR,Denia,Formentera,2024-01-07 20:00:00,2024-01-07,20:00,6,1,...,733.0,393.0,192.0,142.0,588.70,155.762674,173.30,43.510555,5.75,0.654111
4,LevanteFerries,T0000089,DEN-FOR,Denia,Formentera,2024-01-08 08:00:00,2024-01-08,08:00,0,1,...,872.0,474.0,301.0,122.0,601.20,156.441211,177.30,42.798303,5.75,0.501000


In [7]:
feature_cols = [
    "weekday", "month",
    "capacity_pax",
    "avg_ticket_price", "price_index",
    "is_holiday_proxy", "sea_bad_proxy",
    "pax_lag_1w", "pax_lag_2w",
    "pax_roll_mean_20", "pax_roll_std_20",
    "delay_roll_mean_20",
]

X = df[feature_cols]
y = df["pax_real"]

model_final = XGBRegressor(
    n_estimators=700,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

model_final.fit(X, y)
print("✅ model_final entrenado con todo el histórico")

✅ model_final entrenado con todo el histórico


In [8]:
routes = sorted(df["route_id"].unique().tolist())
dep_times = sorted(df["dep_time"].unique().tolist())  # usa las horas reales del dataset

horizon_days = 30

last_date = df["departure_datetime_local"].max().normalize()
start_date = last_date + pd.Timedelta(days=1)
end_date = start_date + pd.Timedelta(days=horizon_days)

future_days = pd.date_range(start=start_date, end=end_date, freq="D")

rows = []
for d in future_days:
    for r in routes:
        # ejemplo: 3 salidas por día (si quieres 4, quita el [:3])
        for t in dep_times[:3]:
            dt = pd.Timestamp(f"{d.date()} {t}")
            rows.append({"route_id": r, "departure_datetime_local": dt, "dep_time": t})

future = pd.DataFrame(rows)
future["weekday"] = future["departure_datetime_local"].dt.weekday
future["month"] = future["departure_datetime_local"].dt.month
future["date"] = future["departure_datetime_local"].dt.normalize()

future.head(), future.shape

(  route_id departure_datetime_local dep_time  weekday  month       date
 0  DEN-FOR      2026-01-01 08:00:00    08:00        3      1 2026-01-01
 1  DEN-FOR      2026-01-01 12:00:00    12:00        3      1 2026-01-01
 2  DEN-FOR      2026-01-01 17:00:00    17:00        3      1 2026-01-01
 3  DEN-IBZ      2026-01-01 08:00:00    08:00        3      1 2026-01-01
 4  DEN-IBZ      2026-01-01 12:00:00    12:00        3      1 2026-01-01,
 (372, 6))

In [9]:
route_defaults = (
    df.groupby("route_id")
    .agg(
        capacity_pax=("capacity_pax","median"),
        avg_ticket_price=("avg_ticket_price","mean"),
        price_index=("price_index","mean"),
        is_holiday_proxy=("is_holiday_proxy","mean"),
        sea_bad_proxy=("sea_bad_proxy","mean"),
    )
    .reset_index()
)

future = future.merge(route_defaults, on="route_id", how="left")

# Redondeos razonables
future["capacity_pax"] = future["capacity_pax"].round().astype(int)
future["avg_ticket_price"] = future["avg_ticket_price"].astype(float)
future["price_index"] = future["price_index"].astype(float)

future.head()

,route_id,departure_datetime_local,dep_time,weekday,month,date,capacity_pax,avg_ticket_price,price_index,is_holiday_proxy,sea_bad_proxy
0,DEN-FOR,2026-01-01 08:00:00,08:00,3,1,2026-01-01,900,53.255390,1.046556,0.033646,0.127934
1,DEN-FOR,2026-01-01 12:00:00,12:00,3,1,2026-01-01,900,53.255390,1.046556,0.033646,0.127934
2,DEN-FOR,2026-01-01 17:00:00,17:00,3,1,2026-01-01,900,53.255390,1.046556,0.033646,0.127934
3,DEN-IBZ,2026-01-01 08:00:00,08:00,3,1,2026-01-01,900,53.290872,1.045428,0.033646,0.127934
4,DEN-IBZ,2026-01-01 12:00:00,12:00,3,1,2026-01-01,900,53.290872,1.045428,0.033646,0.127934


In [10]:
# Creamos clave de slot
df_slot = df.copy()
df_slot["slot_key"] = df_slot["route_id"].astype(str) + "|" + df_slot["weekday"].astype(str) + "|" + df_slot["dep_time"].astype(str)

future["slot_key"] = future["route_id"].astype(str) + "|" + future["weekday"].astype(str) + "|" + future["dep_time"].astype(str)

# Último valor conocido por slot (lag_1w proxy) y uno anterior (lag_2w proxy)
# Nota: en datos reales harías calendar-lag exacto (D-7, D-14). Aquí usamos “últimos registros del slot”.
last_two = (
    df_slot.sort_values(["slot_key","departure_datetime_local"])
          .groupby("slot_key")["pax_real"]
          .apply(lambda s: s.tail(2).values.tolist())
)

def get_last(vals, idx_from_end):
    if not isinstance(vals, list):
        return np.nan
    if len(vals) < idx_from_end:
        return np.nan
    return vals[-idx_from_end]

future["pax_lag_1w"] = future["slot_key"].map(lambda k: get_last(last_two.get(k, np.nan), 1))
future["pax_lag_2w"] = future["slot_key"].map(lambda k: get_last(last_two.get(k, np.nan), 2))

# Si algún slot no existe (raro), fallback a promedio por ruta
route_mean_pax = df.groupby("route_id")["pax_real"].mean().to_dict()
future["pax_lag_1w"] = future["pax_lag_1w"].fillna(future["route_id"].map(route_mean_pax))
future["pax_lag_2w"] = future["pax_lag_2w"].fillna(future["route_id"].map(route_mean_pax))

future[["route_id","dep_time","weekday","pax_lag_1w","pax_lag_2w"]].head(10)

,route_id,dep_time,weekday,pax_lag_1w,pax_lag_2w
0,DEN-FOR,08:00,3,477,373
1,DEN-FOR,12:00,3,658,490
2,DEN-FOR,17:00,3,518,496
3,DEN-IBZ,08:00,3,614,555
4,DEN-IBZ,12:00,3,699,613
5,DEN-IBZ,17:00,3,630,585
6,DEN-PMI,08:00,3,479,475
7,DEN-PMI,12:00,3,581,433
8,DEN-PMI,17:00,3,526,552
9,VAL-IBZ,08:00,3,378,428


In [11]:
roll_by_route = (
    df.sort_values(["route_id","departure_datetime_local"])
      .groupby("route_id")
      .tail(20)
      .groupby("route_id")
      .agg(
          pax_roll_mean_20=("pax_real","mean"),
          pax_roll_std_20=("pax_real","std"),
          delay_roll_mean_20=("delay_minutes","mean")
      )
      .reset_index()
)

future = future.merge(roll_by_route, on="route_id", how="left")

# Fallback si algo queda NaN
future["pax_roll_mean_20"] = future["pax_roll_mean_20"].fillna(future["route_id"].map(route_mean_pax))
future["pax_roll_std_20"] = future["pax_roll_std_20"].fillna(0.0)
future["delay_roll_mean_20"] = future["delay_roll_mean_20"].fillna(df["delay_minutes"].mean())

future.head()

,route_id,departure_datetime_local,dep_time,weekday,month,date,capacity_pax,avg_ticket_price,price_index,is_holiday_proxy,sea_bad_proxy,slot_key,pax_lag_1w,pax_lag_2w,pax_roll_mean_20,pax_roll_std_20,delay_roll_mean_20
0,DEN-FOR,2026-01-01 08:00:00,08:00,3,1,2026-01-01,900,53.255390,1.046556,0.033646,0.127934,DEN-FOR|3|08:00,477,373,566.30,93.736529,8.0
1,DEN-FOR,2026-01-01 12:00:00,12:00,3,1,2026-01-01,900,53.255390,1.046556,0.033646,0.127934,DEN-FOR|3|12:00,658,490,566.30,93.736529,8.0
2,DEN-FOR,2026-01-01 17:00:00,17:00,3,1,2026-01-01,900,53.255390,1.046556,0.033646,0.127934,DEN-FOR|3|17:00,518,496,566.30,93.736529,8.0
3,DEN-IBZ,2026-01-01 08:00:00,08:00,3,1,2026-01-01,900,53.290872,1.045428,0.033646,0.127934,DEN-IBZ|3|08:00,614,555,682.25,109.621778,8.2
4,DEN-IBZ,2026-01-01 12:00:00,12:00,3,1,2026-01-01,900,53.290872,1.045428,0.033646,0.127934,DEN-IBZ|3|12:00,699,613,682.25,109.621778,8.2


In [12]:
X_future = future[feature_cols]
future["pax_pred_ml"] = model_final.predict(X_future)

# Cap por capacidad (regla de negocio)
future["pax_pred_ml"] = np.minimum(future["pax_pred_ml"], future["capacity_pax"]).clip(0)

future["occ_pred_ml"] = future["pax_pred_ml"] / future["capacity_pax"]

future["flag"] = np.select(
    [
        future["occ_pred_ml"] >= 0.92,
        future["occ_pred_ml"].between(0.80, 0.919999),
        future["occ_pred_ml"] < 0.60
    ],
    ["ROJO","ÁMBAR","VERDE"],
    default="NORMAL"
)

# (opcional) acciones sugeridas
future["action"] = np.select(
    [future["flag"].eq("ROJO"), future["flag"].eq("VERDE"), future["flag"].eq("ÁMBAR")],
    [
        "Evaluar refuerzo/cambio buque + ajustar inventario/precio",
        "Activar promo táctica / bundles / campañas por ruta",
        "Monitorizar curva D-14/D-7 y microajustes precio/comunicación"
    ],
    default="Monitor estándar"
)

out_path = f"{PROJECT_ROOT}/data/processed/future_forecast_30d.csv"
future.to_csv(out_path, index=False)

print("Saved ✅", out_path)
future[["route_id","departure_datetime_local","capacity_pax","pax_pred_ml","occ_pred_ml","flag"]].head(20)

Saved ✅ /content/drive/MyDrive/7 Colab Notebooks/1 Porfolio/Balearia/2 Demanda/data/processed/future_forecast_30d.csv


,route_id,departure_datetime_local,capacity_pax,pax_pred_ml,occ_pred_ml,flag
0,DEN-FOR,2026-01-01 08:00:00,900,559.428345,0.621587,NORMAL
1,DEN-FOR,2026-01-01 12:00:00,900,516.272095,0.573636,VERDE
2,DEN-FOR,2026-01-01 17:00:00,900,520.582886,0.578425,VERDE
3,DEN-IBZ,2026-01-01 08:00:00,900,606.442932,0.673825,NORMAL
4,DEN-IBZ,2026-01-01 12:00:00,900,610.486572,0.678318,NORMAL
5,DEN-IBZ,2026-01-01 17:00:00,900,612.277161,0.680308,NORMAL
6,DEN-PMI,2026-01-01 08:00:00,1200,496.638336,0.413865,VERDE
7,DEN-PMI,2026-01-01 12:00:00,1200,495.630585,0.413025,VERDE
8,DEN-PMI,2026-01-01 17:00:00,1200,473.932251,0.394944,VERDE
9,VAL-IBZ,2026-01-01 08:00:00,1200,374.041901,0.311702,VERDE


In [13]:
future["flag"].value_counts()

,count
flag,
VERDE,232
NORMAL,102
ÁMBAR,33
ROJO,5


In [14]:
import os

print("future_forecast_30d.csv:",
      os.path.exists(f"{PROJECT_ROOT}/data/processed/future_forecast_30d.csv"))

print("future_forecast_30d.parquet:",
      os.path.exists(f"{PROJECT_ROOT}/data/processed/future_forecast_30d.parquet"))

future_forecast_30d.csv: True
future_forecast_30d.parquet: False


In [15]:
future.to_parquet(
    f"{PROJECT_ROOT}/data/processed/future_forecast_30d.parquet",
    index=False
)

print("future_forecast_30d.parquet:",
      os.path.exists(f"{PROJECT_ROOT}/data/processed/future_forecast_30d.parquet"))

future_forecast_30d.parquet: True
